# Notebook 04 — synthetic causal model & prediction (the scoreable path)

`causalway.synthetic` generates a 3-class KG (`Patient`/`Hospital`/`Therapy`) from a *known*
ground-truth SEM, so — unlike SCLC — answers here can be **scored against the truth** rather
than only inspected. This isolates two separate error sources: graph error (comparing a model
fitted on the ground-truth graph vs. one fitted on the *discovered* graph) and model error
(comparing either model's answers to the true generative process).

Run with the `rdfenv` kernel. If `results/synthetic/best_GES_constrained.ttl` doesn't exist yet,
run notebook 02 first — the discovered-graph comparisons are skipped gracefully without it.

In [1]:
import os, sys

def _find_root():
    p = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(p, 'causalway')) and os.path.isdir(os.path.join(p, 'algs')):
            return p
        p = os.path.dirname(p)
    raise RuntimeError('project root not found')

PROJECT_ROOT = _find_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import dowhy, pgmpy, sklearn, rdflib, pandas, numpy
print('dowhy', dowhy.__version__, '| pgmpy', pgmpy.__version__, '| sklearn', sklearn.__version__)
print('PROJECT_ROOT =', PROJECT_ROOT)


dowhy 0.14 | pgmpy 1.1.0 | sklearn 1.6.1
PROJECT_ROOT = /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api


## 1. Regenerate the synthetic KG

`runners.run_synthetic.build_synthetic_context` is Plan 1's own helper for this exact KG
(`n_patients=500, n_hospitals=20, max_therapies=4, seed=42`) — reused here so the ground-truth
`EdgeConstraint` (and therefore node order) lines up with `results/synthetic/best_GES_constrained.ttl`
without re-deriving it.

In [2]:
from runners.run_synthetic import build_synthetic_context, SYNTHETIC_TTL
from causalway import synthetic as syn
from causalway.result import OntologicalCausalGraph
from causalway.sources import load_ocg

ctx, truth_adj = build_synthetic_context()  # regenerates kgs/ttls/synthetic_clinic.ttl
print('flat-join shape:', ctx.mat.df.shape, '| multiplicity:', ctx.mat.multiplicity)

ocg_truth = OntologicalCausalGraph.from_discovery(truth_adj, ctx.constraint_kept, method='ground_truth')
print(f'ground truth: {ocg_truth.n} nodes, {int(ocg_truth.adj.sum())} edges')
ocg_truth.to_dataframe()


2026-09-23 09:10:06,227 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/castle/backend/__init__.py[line:36] - INFO: You can use `os.environ['CASTLE_BACKEND'] = backend` to set the backend(`pytorch` or `mindspore`).


2026-09-23 09:10:06,248 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/castle/algorithms/__init__.py[line:36] - INFO: You are using ``pytorch`` as the backend.


/Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Dropped constant columns: ['Patient.receives', 'Patient.treatedAt']
flat-join shape: (1240, 13) | multiplicity: {'Hospital': 62.0, 'Patient': 2.48, 'Therapy': 1.0}
ground truth: 11 nodes, 11 edges


,method,cause,effect,cause_domain,effect_domain,relation,weight
0,ground_truth,Hospital.airPollution,Patient.tumorStage,Hospital,Patient,treatedAt (inverse),1.0
1,ground_truth,Hospital.careQuality,Patient.survival,Hospital,Patient,treatedAt (inverse),1.0
2,ground_truth,Hospital.region,Hospital.airPollution,Hospital,Hospital,ε,1.0
3,ground_truth,Patient.age,Patient.tumorStage,Patient,Patient,ε,1.0
4,ground_truth,Patient.geneticRisk,Patient.tumorStage,Patient,Patient,ε,1.0
5,ground_truth,Patient.smoking,Patient.tumorStage,Patient,Patient,ε,1.0
6,ground_truth,Patient.tumorStage,Patient.survival,Patient,Patient,ε,1.0
7,ground_truth,Patient.tumorStage,Therapy.dosage,Patient,Therapy,receives (forward),1.0
8,ground_truth,Therapy.dosage,Therapy.toxicity,Therapy,Therapy,ε,1.0
9,ground_truth,Therapy.drugClass,Therapy.toxicity,Therapy,Therapy,ε,1.0


In [3]:
DISCOVERED_TTL = os.path.join(PROJECT_ROOT, 'results', 'synthetic', 'best_GES_constrained.ttl')
HAVE_DISCOVERED = os.path.exists(DISCOVERED_TTL)
if HAVE_DISCOVERED:
    ocg_discovered = load_ocg(DISCOVERED_TTL)
    print(f'discovered ({ocg_discovered.method}): {ocg_discovered.n} nodes, '
          f'{int(ocg_discovered.adj.sum())} edges')
else:
    print(f'{DISCOVERED_TTL} not found — run notebook 02 first for the discovered-graph comparisons; '
          'continuing with the ground-truth graph only.')


discovered (None): 11 nodes, 14 edges


## 2. Fit on the ground-truth graph, and on the discovered graph

Every metric below is reported for both, so graph error (structure) and model error
(mechanism fit) stay distinguishable.

In [4]:
from causalway.model import CausalModel

SYNTH_KG = SYNTHETIC_TTL
model_truth = CausalModel.fit(ocg_truth, SYNTH_KG, quality='good', random_state=0)
print('model_truth:', model_truth.model_id)
model_truth.mechanism_table()


Fitting causal models:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.airPollution:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.careQuality:   0%|          | 0/11 [00:00<?, ?it/s] 

Fitting causal mechanism of node Hospital.region:   0%|          | 0/11 [00:00<?, ?it/s]     

Fitting causal mechanism of node Patient.age:   0%|          | 0/11 [00:00<?, ?it/s]    

...

Fitting causal mechanism of node Therapy.drugClass:  64%|██████▎   | 7/11 [00:01<00:00,  5.17it/s]

Fitting causal mechanism of node Therapy.toxicity:  64%|██████▎   | 7/11 [00:01<00:00,  5.17it/s] 

Fitting causal mechanism of node Therapy.toxicity: 100%|██████████| 11/11 [00:01<00:00,  7.92it/s]

model_truth: ground_truth-92c943f586de-92c943f5


,node,dtype,is_root,mechanism_type,invertible
0,Hospital.airPollution,categorical,False,InvertibleClassifierFCM,True
1,Hospital.careQuality,categorical,True,EmpiricalDistribution,True
2,Hospital.region,categorical,True,EmpiricalDistribution,True
3,Patient.age,categorical,True,EmpiricalDistribution,True
4,Patient.geneticRisk,categorical,True,EmpiricalDistribution,True
5,Patient.smoking,categorical,True,EmpiricalDistribution,True
6,Patient.survival,categorical,False,InvertibleClassifierFCM,True
7,Patient.tumorStage,categorical,False,InvertibleClassifierFCM,True
8,Therapy.dosage,categorical,False,InvertibleClassifierFCM,True
9,Therapy.drugClass,categorical,True,EmpiricalDistribution,True


GES's output isn't guaranteed acyclic (Plan 1 §3.2): the constrained-GES run that produced
`best_GES_constrained.ttl` returned a `Hospital.region <-> Hospital.airPollution` 2-cycle on top
of the 11 ground-truth edges. `on_cycle='strict'` (the default) raises rather than silently
pick a direction; `'weight'` breaks each cycle by dropping its lowest-`cw:weight` edge instead —
the deliberate choice made here, not the silent one.

In [5]:
if HAVE_DISCOVERED:
    model_discovered = CausalModel.fit(ocg_discovered, SYNTH_KG, quality='good', random_state=0,
                                       on_missing='drop', on_cycle='weight')
    print('model_discovered:', model_discovered.model_id)
    print(model_discovered.mechanism_table())
else:
    model_discovered = None


Fitting causal models:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.airPollution:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.careQuality:   0%|          | 0/11 [00:00<?, ?it/s] 

Fitting causal mechanism of node Hospital.region:   0%|          | 0/11 [00:00<?, ?it/s]     

Fitting causal mechanism of node Patient.age:   0%|          | 0/11 [00:00<?, ?it/s]    

...

Fitting causal mechanism of node Therapy.dosage:  73%|███████▎  | 8/11 [00:03<00:01,  2.09it/s]    

Fitting causal mechanism of node Therapy.drugClass:  73%|███████▎  | 8/11 [00:03<00:01,  2.09it/s]

Fitting causal mechanism of node Therapy.toxicity:  73%|███████▎  | 8/11 [00:03<00:01,  2.09it/s] 

Fitting causal mechanism of node Therapy.toxicity: 100%|██████████| 11/11 [00:03<00:00,  3.35it/s]

model_discovered: ocg-15078d93349b-d412ec0f
                     node        dtype  is_root           mechanism_type  \
0   Hospital.airPollution  categorical    False  InvertibleClassifierFCM   
1    Hospital.careQuality  categorical    False  InvertibleClassifierFCM   
2         Hospital.region  categorical     True    EmpiricalDistribution   
3             Patient.age  categorical    False  InvertibleClassifierFCM   
4     Patient.geneticRisk  categorical     True    EmpiricalDistribution   
5         Patient.smoking  categorical     True    EmpiricalDistribution   
6        Patient.survival  categorical    False  InvertibleClassifierFCM   
7      Patient.tumorStage  categorical    False  InvertibleClassifierFCM   
8          Therapy.dosage  categorical    False  InvertibleClassifierFCM   
9       Therapy.drugClass  categorical     True    EmpiricalDistribution   
10       Therapy.toxicity  categorical    False  InvertibleClassifierFCM   

    invertible  
0         True  
1        

## 3. Evaluate + falsify — which graph is fit to answer queries?

This is the notebook 03 -> notebook 04 feedback loop Plan 2 §7 describes: falsification here
has ground truth to check itself against, not just internal consistency.

In [6]:
from causalway import evaluation

cv_truth = evaluation.held_out_cv(model_truth, n_splits=5, random_state=0)
cv_truth['graph'] = 'ground_truth'
frames = [cv_truth]
if model_discovered is not None:
    cv_disc = evaluation.held_out_cv(model_discovered, n_splits=5, random_state=0)
    cv_disc['graph'] = 'discovered'
    frames.append(cv_disc)
import pandas as pd
pd.concat(frames, ignore_index=True).sort_values(['node', 'graph'])


,node,metric,model,baseline,beats_baseline,macro_f1,graph
5,Hospital.airPollution,accuracy,0.843548,0.442742,True,0.866406,discovered
0,Hospital.airPollution,accuracy,0.843548,0.442742,True,0.866406,ground_truth
6,Hospital.careQuality,accuracy,0.669355,0.669355,False,0.400966,discovered
7,Patient.age,accuracy,0.437903,0.307258,True,0.345763,discovered
8,Patient.survival,accuracy,0.735484,0.474194,True,0.707846,discovered
1,Patient.survival,accuracy,0.735484,0.474194,True,0.707846,ground_truth
9,Patient.tumorStage,accuracy,0.631452,0.354839,True,0.596752,discovered
2,Patient.tumorStage,accuracy,0.678226,0.354839,True,0.619817,ground_truth
10,Therapy.dosage,accuracy,0.742742,0.516129,True,0.675045,discovered
3,Therapy.dosage,accuracy,0.742742,0.516129,True,0.675045,ground_truth


In [7]:
RUN_FALSIFY = False  # kernel CI tests over the flat join; opt-in, can take a while
if RUN_FALSIFY:
    print('ground truth graph:')
    print(evaluation.falsify(model_truth, n_permutations=20, show_progress_bar=True))
    if model_discovered is not None:
        print('discovered graph:')
        print(evaluation.falsify(model_discovered, n_permutations=20, show_progress_bar=True))
else:
    print('Skipped — set RUN_FALSIFY = True to run gcm.falsify.falsify_graph on both graphs.')


Skipped — set RUN_FALSIFY = True to run gcm.falsify.falsify_graph on both graphs.


## 4. A ground-truth simulator

`causalway.synthetic`'s per-variable sampling functions (`_sample_stage`, `_sample_survival`, ...)
implement the *exact* structural equations `generate()` used, just without exposing a
forward-simulation entry point. This wraps them into one ancestral sampler that can hold any
variable fixed — the textbook Monte-Carlo definition of both plain conditioning (an unforced
variable, filtered on afterwards) and `do(...)` (a forced variable, propagated through with
fresh noise). It does **not** attempt exact single-row counterfactuals: that needs the specific
noise draw behind one already-generated row, which `generate()` doesn't expose — sections 6-7
use coupling comparison and an aggregation ablation instead, neither of which needs it.

In [8]:
import numpy as np

NODE_OF = {  # short generator variable -> SCM column name
    'age': 'Patient.age', 'smoking': 'Patient.smoking', 'geneticRisk': 'Patient.geneticRisk',
    'tumorStage': 'Patient.tumorStage', 'survival': 'Patient.survival',
    'region': 'Hospital.region', 'airPollution': 'Hospital.airPollution', 'careQuality': 'Hospital.careQuality',
    'drugClass': 'Therapy.drugClass', 'dosage': 'Therapy.dosage', 'toxicity': 'Therapy.toxicity',
}

def simulate_true(n=20_000, seed=0, do=None):
    """Ancestral Monte-Carlo sample from the true SEM (one therapy per simulated patient).

    `do` maps short variable names (keys of NODE_OF) to a forced value; every other
    variable is drawn from the same structural equation `generate()` uses, with fresh
    noise. Returns a DataFrame keyed by the *short* names.
    """
    do = do or {}
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n):
        age = do.get('age', str(rng.choice(syn.AGE_VALUES)))
        smoking = do.get('smoking', str(rng.choice(syn.SMOKING_VALUES)))
        gen = do.get('geneticRisk', str(rng.choice(syn.GEN_VALUES)))
        region = do.get('region', str(rng.choice(syn.REGION_VALUES)))
        air = do.get('airPollution', syn._sample_air(rng, region))
        care = do.get('careQuality', syn._sample_care(rng))
        stage = do.get('tumorStage', syn._sample_stage(rng, age, smoking, gen, air))
        drug = do.get('drugClass', str(rng.choice(syn.DRUG_VALUES)))
        dosage = do.get('dosage', syn._sample_dosage(rng, stage))
        tox = do.get('toxicity', syn._sample_tox(rng, drug, dosage))
        tox_score = syn._tox_score[tox]
        survival = do.get('survival', syn._sample_survival(rng, stage, care, tox_score))
        rows.append(dict(age=age, smoking=smoking, geneticRisk=gen, region=region,
                          airPollution=air, careQuality=care, tumorStage=stage,
                          drugClass=drug, dosage=dosage, toxicity=tox, survival=survival))
    return pd.DataFrame(rows)

def true_distribution(target_short: str, n=20_000, seed=0, do=None, evidence=None):
    """P(target | do(...)) via forcing, or P(target | evidence) via rejection filtering."""
    df = simulate_true(n=n, seed=seed, do=do)
    if evidence:
        mask = pd.Series(True, index=df.index)
        for k, v in evidence.items():
            mask &= (df[k] == v)
        df = df[mask]
    counts = df[target_short].value_counts(normalize=True)
    return counts.to_dict(), len(df)

_ = simulate_true(n=5)  # smoke check
print('simulator OK')


simulator OK


## 5. Conditional prediction vs. the true SEM

In [9]:
true_dist, n_kept = true_distribution('tumorStage', n=50_000, evidence={'smoking': 'Yes'})
print(f'true P(tumorStage | smoking=Yes), n={n_kept}:', {k: round(v, 3) for k, v in true_dist.items()})

ans_truth = model_truth.condition('Patient.tumorStage', {'Patient.smoking': 'Yes'})
print('model_truth:     ', {k: round(v, 3) for k, v in ans_truth.distribution.items()}, f'(backend={ans_truth.backend})')

if model_discovered is not None and 'Patient.tumorStage' in model_discovered.spec.columns:
    ans_disc = model_discovered.condition('Patient.tumorStage', {'Patient.smoking': 'Yes'})
    print('model_discovered:', {k: round(v, 3) for k, v in ans_disc.distribution.items()}, f'(backend={ans_disc.backend})')


true P(tumorStage | smoking=Yes), n=24843: {'IV': 0.381, 'III': 0.367, 'II': 0.213, 'I': 0.039}
model_truth:      {'I': 0.04, 'II': 0.159, 'III': 0.348, 'IV': 0.454} (backend=pgmpy-exact)
model_discovered: {'I': 0.04, 'II': 0.159, 'III': 0.349, 'IV': 0.452} (backend=pgmpy-exact)


## 6. Interventional: estimated vs. true ACE for the cross-class edges

`Hospital.airPollution -> Patient.tumorStage` (via `treatedAt`, inverse direction) and
`Hospital.careQuality -> Patient.survival` (also inverse) are exactly the edges E6/§10 risk #2
discuss: the flat-join SCM reaches them only through the shared column in every joined row.

In [10]:
def compare_intervention(node_short, target_short, value, n_true=50_000):
    node, target = NODE_OF[node_short], NODE_OF[target_short]
    true_dist, _ = true_distribution(target_short, n=n_true, do={node_short: value})
    ans_truth = model_truth.intervene({node: value}, target=target, num_samples=10_000)
    rows = {'true': true_dist, 'model_truth': ans_truth.distribution}
    if model_discovered is not None and node in model_discovered.spec.columns and target in model_discovered.spec.columns:
        ans_disc = model_discovered.intervene({node: value}, target=target, num_samples=10_000)
        rows['model_discovered'] = ans_disc.distribution
    return pd.DataFrame(rows).round(3)

compare_intervention('airPollution', 'tumorStage', 'High')


,true,model_truth,model_discovered
IV,0.503,0.534,0.533
III,0.375,0.362,0.362
II,0.115,0.097,0.106
I,0.006,0.007,NaN


In [11]:
compare_intervention('careQuality', 'survival', 'High')


,true,model_truth,model_discovered
Short,0.399,0.419,0.415
Medium,0.304,0.332,0.322
Long,0.297,0.250,0.262


## 7. Counterfactual: Gumbel-max vs. the ordinal coupling

`tumorStage` (I < II < III < IV) and `survival` (Short < Medium < Long) both have a real order
— `ordinal=` routes them through `DiscreteAdditiveNoiseModel` (invertible out of the box)
instead of Gumbel-max. Refitting and comparing the *same* counterfactual query under both
quantifies how much of the answer is the coupling choice rather than the data (§3.4, §10 risk #1).

In [12]:
model_truth_ordinal = CausalModel.fit(ocg_truth, SYNTH_KG, quality='good', random_state=0,
                                      ordinal=['Patient.tumorStage', 'Patient.survival'])
model_truth_ordinal.mechanism_table().query("node in ['Patient.tumorStage', 'Patient.survival']")


Fitting causal models:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.airPollution:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.careQuality:   0%|          | 0/11 [00:00<?, ?it/s] 

Fitting causal mechanism of node Hospital.region:   0%|          | 0/11 [00:00<?, ?it/s]     

Fitting causal mechanism of node Patient.age:   0%|          | 0/11 [00:00<?, ?it/s]    

...

Fitting causal mechanism of node Therapy.dosage:  82%|████████▏ | 9/11 [00:01<00:00,  4.92it/s]

Fitting causal mechanism of node Therapy.drugClass:  82%|████████▏ | 9/11 [00:01<00:00,  4.92it/s]

Fitting causal mechanism of node Therapy.toxicity:  82%|████████▏ | 9/11 [00:01<00:00,  4.92it/s] 

Fitting causal mechanism of node Therapy.toxicity: 100%|██████████| 11/11 [00:01<00:00,  6.97it/s]

,node,dtype,is_root,mechanism_type,invertible
6,Patient.survival,ordinal,False,DiscreteAdditiveNoiseModel,True
7,Patient.tumorStage,ordinal,False,DiscreteAdditiveNoiseModel,True


In [13]:
patient_col = [c for c in model_truth.spec.mat.entity_ids.columns if 'Patient' in c][0]
entity0 = str(model_truth.spec.mat.entity_ids[patient_col].iloc[0])

for label, m in [('gumbel-max', model_truth), ('ordinal', model_truth_ordinal)]:
    ans = m.counterfactual({(entity0, 'Patient.smoking'): 'Yes'}, entity=entity0,
                           target='Patient.tumorStage', num_samples=200)
    top = max(ans.distribution, key=ans.distribution.get) if ans.distribution else ans.predicted
    print(f'{label:>10}: do(smoking:=Yes) -> tumorStage = {top!r} '
          f'(p={ans.distribution.get(top, float("nan")):.2f}, coupling={ans.coupling})')


gumbel-max: do(smoking:=Yes) -> tumorStage = 'II' (p=1.00, coupling=gumbel-max)
   ordinal: do(smoking:=Yes) -> tumorStage = 'II' (p=1.00, coupling=None)


## 8. Aggregation ablation — single-row vs. entity-aggregated counterfactual

`Patient.survival` truly depends on an *aggregate* over a patient's therapies. A single flat-join
row only sees one therapy, so a single-row counterfactual and the row-aggregated entity-level
counterfactual (what `CausalModel.counterfactual` returns by default) can disagree — this is
the aggregation loss Plan 1 §9.4 / Plan 2 §10 risk #3, quantified rather than hidden.

In [14]:
from causalway.entities import rows_for_entity
from dowhy.gcm._noise import compute_noise_from_data
from dowhy.gcm import counterfactual_samples

patient_col = [c for c in model_truth.spec.mat.entity_ids.columns if 'Patient' in c][0]
counts = model_truth.spec.mat.entity_ids[patient_col].value_counts()
multi_row_patient = counts[counts > 1].index[0]
rows = rows_for_entity(model_truth.spec.mat, multi_row_patient, var=patient_col)
print(f'{multi_row_patient} spans {len(rows)} flat-join rows (one per therapy)')

target = 'Patient.survival'
do_node, do_value = 'Therapy.dosage', model_truth.spec.data['Therapy.dosage'].mode().iloc[0]
fn = {do_node: (lambda _, v=do_value: v)}

# entity-aggregated (all rows) — what CausalModel.counterfactual returns
ans_aggregated = model_truth.counterfactual({(multi_row_patient, do_node): do_value},
                                            entity=multi_row_patient, target=target, num_samples=100)

# single-row (first row only), same intervention, same abduction/evaluation machinery,
# redrawing noise each iteration since every node here is categorical (Gumbel-max, stochastic abduction)
observed_one_row = model_truth.spec.data.loc[[rows[0]], model_truth.spec.columns]
single_row_draws = pd.concat(
    [counterfactual_samples(model_truth.scm, fn, noise_data=compute_noise_from_data(
        model_truth.scm, observed_one_row))[target] for _ in range(100)],
    ignore_index=True)
from causalway.entities import aggregate
dtype = model_truth.spec.dtypes[target]
ans_single_row = aggregate(single_row_draws, dtype)

print('entity-aggregated (all rows):', ans_aggregated.predicted, ans_aggregated.distribution)
print('single-row only:             ', ans_single_row,
      single_row_draws.value_counts(normalize=True).round(3).to_dict())


http://causalkg.example.org/synthetic/patient_79 spans 4 flat-join rows (one per therapy)


entity-aggregated (all rows): Long {'Long': 0.76, 'Medium': 0.24}
single-row only:              Long {'Long': 0.67, 'Medium': 0.33}


## 9. Cross-entity intervention (E6): hospital -> patient spillover

`do(hospital.careQuality := high)` inside a counterfactual query about *one patient* — the
§6.5 reach check needs both a causal path (already shown above) and *relational* reach (the
hospital must actually be this patient's, checked against the source KG).

In [15]:
from causalway.queries import Intervention, Query, validate_query

hospital_col = [c for c in model_truth.spec.mat.entity_ids.columns if 'Hospital' in c][0]
patient_row = rows_for_entity(model_truth.spec.mat, multi_row_patient, var=patient_col)[0]
own_hospital = str(model_truth.spec.mat.entity_ids.loc[patient_row, hospital_col])
other_hospital = next(
    h for h in model_truth.spec.mat.entity_ids[hospital_col].astype(str).unique()
    if h != own_hospital
)

q_own = Query(kind='counterfactual', model_id=model_truth.model_id, target=[target],
             interventions=[Intervention(node='Hospital.careQuality', value='High', entity=own_hospital)],
             entity=multi_row_patient)
validate_query(q_own, model_truth.spec.ocg, model_truth.spec.mat)
print("validate_query accepted the patient's own hospital ✓")

q_other = Query(kind='counterfactual', model_id=model_truth.model_id, target=[target],
                interventions=[Intervention(node='Hospital.careQuality', value='High', entity=other_hospital)],
                entity=multi_row_patient)
try:
    validate_query(q_other, model_truth.spec.ocg, model_truth.spec.mat)
    print('unexpectedly accepted an unrelated hospital')
except ValueError as e:
    print('validate_query correctly rejected an unrelated hospital:', e)

ans_cf = model_truth.counterfactual({(own_hospital, 'Hospital.careQuality'): 'High'},
                                    entity=multi_row_patient, target=target, num_samples=100)
print(f'\ncounterfactual {target} for {multi_row_patient} under do(own hospital careQuality:=High):',
      ans_cf.distribution)


validate_query accepted the patient's own hospital ✓
validate_query correctly rejected an unrelated hospital: validate_query: http://causalkg.example.org/synthetic/hospital_1 is not joined to http://causalkg.example.org/synthetic/patient_79 in the source KG along any relation (§6.5 relational reach).

counterfactual Patient.survival for http://causalkg.example.org/synthetic/patient_79 under do(own hospital careQuality:=High): {'Long': 1.0}


The same intervention as a **population** `InterventionalQuery` shows the spillover across every
patient joined to that hospital — no `aboutEntity`, no `onEntity` on the intervention (§6.7.1).

In [16]:
ans_population = model_truth.intervene({'Hospital.careQuality': 'High'}, target=target,
                                        reference={'Hospital.careQuality': 'Low'}, num_samples=10_000)
print('population-level do(careQuality:=High) vs do(careQuality:=Low):')
print('distribution:', ans_population.distribution)
print('per-level contrast:', ans_population.effect)


population-level do(careQuality:=High) vs do(careQuality:=Low):
distribution: {'Short': 0.4218, 'Medium': 0.3229, 'Long': 0.2553}
per-level contrast: {'Long': 0.003300000000000025, 'Medium': -0.008799999999999975, 'Short': 0.005500000000000005}


## 10. (Optional) Endpoint path

Materialise the same KG from a SPARQL endpoint instead of the file, if one is reachable
(Plan 2 open question Q4 names a GraphDB instance for `synthetic_clinic.ttl`). Skipped cleanly
when unreachable — this is a connectivity check, not a fitting run.

In [17]:
ENDPOINT = 'http://localhost:7200/repositories/syn_clinic'
try:
    from causalway.sources import resolve_schema
    schema_ep = resolve_schema(ENDPOINT, timeout=5)
    print(schema_ep.summary())
except Exception as e:
    print(f'Endpoint unreachable ({type(e).__name__}: {e}) — skipping. '
          'This cell only matters if you have a local Fuseki/GraphDB serving this KG.')


resolve_schema: infer_missing defaults to False for endpoints (pass infer_missing=True to run the T-Box inference queries against it anyway).


{'n_classes': 5, 'n_object_properties': 2, 'n_object_domain_range': 2, 'n_data_properties': 11, 'n_data_domain_range': 11, 'n_inferred': 0}


## Save both models

In [18]:
for m in [model_truth, model_discovered]:
    if m is None:
        continue
    out_dir = os.path.join(PROJECT_ROOT, 'results', 'models', m.model_id)
    m.save(out_dir)
    print('saved', m.model_id, '->', out_dir)


saved ground_truth-92c943f586de-92c943f5 -> /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/results/models/ground_truth-92c943f586de-92c943f5


saved ocg-15078d93349b-d412ec0f -> /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/results/models/ocg-15078d93349b-d412ec0f
